In [ ]:
#Importation et nettoyage des données

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

df = pd.read_csv('global_power_plant_database.csv', low_memory=False)

print("--- Aperçu des données ---")
print(df.head())

print("\n--- Valeurs manquantes par colonne ---")
print(df.isnull().sum())

df = df.dropna(subset=['capacity_mw', 'primary_fuel', 'country_long'])


if 'commissioning_year' in df.columns:
    median_year = df['commissioning_year'].median()
    df['commissioning_year'] = df['commissioning_year'].fillna(median_year)

df['capacity_mw'] = np.asarray(df['capacity_mw'], dtype=np.float64)


#Analyse statistique
print("\n--- Statistiques clés des colonnes numériques ---")
print(df[['capacity_mw']].describe())

print("\n--- Top 10 des pays avec le plus de centrales ---")
print(df['country_long'].value_counts().head(10))

print("\n--- Répartition par type de combustible (Nombre de centrales) ---")
print(df['primary_fuel'].value_counts())

capacities = df['capacity_mw'].to_numpy()
print(f"\nPuissance Moyenne (NumPy): {np.mean(capacities):.2f} MW")
print(f"Puissance Médiane (NumPy): {np.median(capacities):.2f} MW")
print(f"Écart-type (NumPy): {np.std(capacities):.2f} MW")

cap_nuclear = df[df['primary_fuel'] == 'Nuclear']['capacity_mw'].dropna().to_numpy()
cap_hydro = df[df['primary_fuel'] == 'Hydro']['capacity_mw'].dropna().to_numpy()

t_stat, p_val = stats.ttest_ind(cap_nuclear, cap_hydro, equal_var=False)
print("\n--- Test d'hypothèse (Nucléaire vs Hydro) ---")
print(f"Statistique T : {t_stat:.4f}")
print(f"P-value : {p_val:.4e}")

if p_val < 0.05:
    print("👉 Rejet de l'hypothèse nulle : La différence de puissance moyenne est statistiquement significative.")
else:
    print("👉 Échec du rejet de l'hypothèse nulle : Pas de différence statistiquement significative.")

#  Analyse des séries chronologiques
if 'commissioning_year' in df.columns:
    df_time = df[(df['commissioning_year'] >= 1900) & (df['commissioning_year'] <= 2026)]
    
    yearly_capacity = df_time.groupby('commissioning_year')['capacity_mw'].sum()
    
    df_time['decade'] = (df_time['commissioning_year'] // 10) * 10
    fuel_evolution = df_time.pivot_table(index='decade', columns='primary_fuel', values='capacity_mw', aggfunc='sum').fillna(0)
    
    print("\n--- Évolution de la capacité par décennie et par carburant (Aperçu) ---")
    print(fuel_evolution.tail())

    plt.figure(figsize=(14, 10))

#Visualisation avancée
plt.subplot(2, 1, 1)
fuel_capacity = df.groupby('primary_fuel')['capacity_mw'].sum().sort_values(ascending=False)
sns.barplot(x=fuel_capacity.values, y=fuel_capacity.index, palette='viridis')
plt.title('Capacité totale installée dans le monde par type de combustible (MW)')
plt.xlabel('Capacité totale (MW)')

plt.subplot(2, 1, 2)
if 'longitude' in df.columns and 'latitude' in df.columns:
    sns.scatterplot(data=df, x='longitude', y='latitude', hue='primary_fuel', alpha=0.5, palette='tab20', size='capacity_mw', sizes=(1, 200))
    plt.title('Répartition géographique des centrales électriques mondiales')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Combustibles')

plt.tight_layout()
plt.show()
#Opérations matricielles en contexte réel
print("\n--- Opérations Matricielles & Analyse de la Variance ---")


features = df[['capacity_mw', 'latitude', 'longitude']].dropna()
X = features.to_numpy()
X_scaled = (X - np.mean(X, axis=0)) / np.std(X, axis=0)

cov_matrix = np.cov(X_scaled, rowvar=False)
print("Matrice de covariance :\n", cov_matrix)

eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
print("\nValeurs propres :", eigenvalues)
print("Vecteurs propres :\n", eigenvectors)

print("\n> Discussion pour le rapport :")
print("Les valeurs propres indiquent la quantité de variance capturée par chaque composante principale.")
print("Le vecteur propre associé à la plus grande valeur propre montre la direction d'alignement maximum de nos données")
print("(par exemple, si la capacité est fortement liée à une zone géographique spécifique).")




